# AI Pullback Prediction — Kaggle GPU (otomatis)

Notebook ini **mengunduh data XAUUSD sendiri dari Dukascopy**, memvalidasi bid/ask, menyiapkan Parquet, lalu melatih LSTM/Transformer di GPU Kaggle. Tidak perlu upload dataset manual.

Aktifkan **Settings → Accelerator → GPU** dan **Internet = On**. Default unduhan 2015-01-01 s.d. 2024-01-01 (tanggal akhir eksklusif): mencakup train 2015–2020, early stopping 2021, kalibrasi 2022, evaluasi 2023. Data 2024+ tidak diunduh atau digunakan.


In [ ]:
import os, sys, subprocess, shutil, json
from pathlib import Path
REPO_URL = 'https://github.com/NodePro-gif/AI-Pullback-Prediction-.git'
REPO_DIR = Path('/kaggle/working/AI-Pullback-Prediction-')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[deep-learning]'], cwd=REPO_DIR, check=True)
print('Repo dan PyTorch terpasang.')


In [ ]:
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU tidak terdeteksi. Aktifkan Settings → Accelerator → GPU dan restart session.')
print('GPU:', torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB')
torch.backends.cudnn.benchmark = True


## 1. Atur periode dan model

Untuk M5, ubah `TIMEFRAME = 'M5'`. Jalankan satu timeframe per notebook. Unduhan M1 beberapa tahun berukuran besar dan dapat memerlukan waktu lama; file mentah disimpan di `/kaggle/working` agar bisa dipakai ulang selama session yang sama.


In [ ]:
# Periode development saja. End date eksklusif; 2024+ tidak diunduh.
DOWNLOAD_START = '2015-01-01'
DOWNLOAD_END = '2024-01-01'
TIMEFRAME = 'M1'  # M1 atau M5
ARCHITECTURE = 'lstm'  # lstm atau transformer
SEQUENCE_LENGTH = 128
EPOCHS = 30
BATCH_SIZE = 256
HIDDEN_DIM = 96
NUM_LAYERS = 2
LEARNING_RATE = 1e-3
PATIENCE = 5
if TIMEFRAME not in {'M1', 'M5'} or ARCHITECTURE not in {'lstm', 'transformer'}:
    raise ValueError('TIMEFRAME: M1/M5; ARCHITECTURE: lstm/transformer')
RAW_DIR = Path('/kaggle/working/data/raw')
PROCESSED_DIR = Path('/kaggle/working/data/processed')
ARTIFACT_DIR = Path('/kaggle/working/artifacts')
RAW_DIR.mkdir(parents=True, exist_ok=True); PROCESSED_DIR.mkdir(parents=True, exist_ok=True); ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
BID = RAW_DIR / f'xauusd-{TIMEFRAME.lower()}-bid-{DOWNLOAD_START}-{DOWNLOAD_END}.csv'
ASK = RAW_DIR / f'xauusd-{TIMEFRAME.lower()}-ask-{DOWNLOAD_START}-{DOWNLOAD_END}.csv'
PARQUET = PROCESSED_DIR / f'XAUUSD_{TIMEFRAME}_{DOWNLOAD_START}_{DOWNLOAD_END}.parquet'
MODEL = ARTIFACT_DIR / f'xauusd_{TIMEFRAME.lower()}_{ARCHITECTURE}.joblib'
print('Raw:', RAW_DIR, '| Processed:', PARQUET, '| Model:', MODEL)


## 2. Unduh otomatis dari Dukascopy dan siapkan dataset

Memerlukan internet Kaggle aktif. Downloader mengambil bid dan ask terpisah, memakai cache/retry, lalu loader menyelaraskan timestamp dan memvalidasi OHLC. Jika proses terputus, jalankan ulang sel; cache downloader dipertahankan.


In [ ]:
# Node.js/npm dibutuhkan oleh dukascopy-node. Pasang bila image Kaggle belum menyediakannya.
if not shutil.which('npx'):
    if os.geteuid() == 0:
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'nodejs', 'npm'], check=True)
    else:
        raise RuntimeError('npx tidak tersedia dan notebook bukan root; aktifkan image Kaggle dengan Node.js.')
print(subprocess.run(['node', '--version'], capture_output=True, text=True, check=True).stdout.strip())
print(subprocess.run(['npm', '--version'], capture_output=True, text=True, check=True).stdout.strip())

if not (BID.exists() and ASK.exists()):
    cmd = [sys.executable, '-m', 'pullback_prediction.cli', 'download-data', '--from-date', DOWNLOAD_START, '--to-date', DOWNLOAD_END, '--timeframe', TIMEFRAME, '--output-dir', str(RAW_DIR), '--batch-size', '5', '--batch-pause-ms', '1000', '--retries', '5']
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
else:
    print('Raw bid/ask sudah ada; unduhan dilewati.')

if not PARQUET.exists():
    cmd = [sys.executable, '-m', 'pullback_prediction.cli', 'prepare-data', '--bid-input', str(BID), '--ask-input', str(ASK), '--timeframe', TIMEFRAME, '--output', str(PARQUET)]
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
else:
    print('Parquet sudah ada; persiapan data dilewati.')
print('Dataset siap:', PARQUET, '| size GiB:', round(PARQUET.stat().st_size / 1024**3, 3))


## 3. Training GPU

Training memakai data 2015–2020; 2021 untuk early stopping; 2022 untuk kalibrasi; 2023 untuk evaluasi. CUDA mixed precision aktif. Jika CUDA OOM, turunkan `BATCH_SIZE` menjadi 128, 64, atau 32 lalu jalankan ulang sel training.


In [ ]:
cmd = [sys.executable, '-m', 'pullback_prediction.cli', 'train-sequence', '--input', str(PARQUET), '--output', str(MODEL), '--architecture', ARCHITECTURE, '--timeframe', TIMEFRAME, '--sequence-length', str(SEQUENCE_LENGTH), '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--hidden-dim', str(HIDDEN_DIM), '--num-layers', str(NUM_LAYERS), '--learning-rate', str(LEARNING_RATE), '--patience', str(PATIENCE), '--device', 'cuda', '--mixed-precision', '--num-workers', '2']
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
metrics_path = MODEL.with_suffix('.metrics.json')
metrics = json.loads(metrics_path.read_text())
print(json.dumps(metrics, indent=2))
print('Artifacts:')
for p in sorted(ARTIFACT_DIR.glob('*')):
    print(p.name, f'{p.stat().st_size / 1024**2:.2f} MiB')


## 4. Simpan hasil

File ada di `/kaggle/working/artifacts/`. Gunakan panel Files Kaggle untuk membuat output dataset atau mengunduhnya. Simpan `.joblib`, `.pt`, dan `.metrics.json` bersama-sama. Data mentah berada di `/kaggle/working/data/raw/`.

Penting: unduhan historis M1 9 tahun bisa memakan waktu dan ruang disk besar. Kaggle session memiliki batas waktu dan penyimpanan; jika unduhan tidak selesai, cache lokal sesi membantu melanjutkan selama file kerja masih tersedia. Notebook tidak mengklaim profit atau win rate tertentu; evaluasi hasil dan backtest wajib dilakukan.
